# LeetCode 917 — Reverse Only Letters bằng Python

**Link đề:** https://leetcode.com/problems/reverse-only-letters/
**Độ khó:** Easy · **Chủ đề:** Two Pointers (hai con trỏ), String

```python
class Solution:
    def reverseOnlyLetters(self, s: str) -> str:
        ...
```

## Mục tiêu học
1. Hiểu đề: đảo ngược **chỉ các chữ cái**, ký tự khác **đứng yên**.
2. Nắm nền tảng: chuỗi bất biến (immutable), `list(s)`, `"".join(...)`, `str.isalpha()`, slicing `[::-1]`.
3. Đi từ cách trực giác (gom chữ cái rồi đặt lại) → stack → **hai con trỏ** hoán đổi tại chỗ.
4. Trace từng bước, chứng minh tính đúng bằng invariant, phân tích Time/Space.
5. Nhận ra pattern "đảo ngược có bỏ qua" để áp dụng cho bài 345 Reverse Vowels, 125 Valid Palindrome...

**Cách học:** đọc markdown → đoán output cell code → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài — đảo ngược nhưng giữ nguyên ký tự không phải chữ

Cho chuỗi `s`. Trả về chuỗi mới sao cho:
- Mọi ký tự **không phải chữ cái tiếng Anh** giữ nguyên vị trí.
- Mọi **chữ cái** (hoa hoặc thường) được đảo ngược thứ tự.

| `s` | Output |
|---|---|
| `"ab-cd"` | `"dc-ba"` |
| `"a-bC-dEf-ghIj"` | `"j-Ih-gfE-dCba"` |
| `"Test1ng-Leet=code-Q!"` | `"Qedo1ct-eeLg=ntse-T!"` |

**Constraints:** `1 <= s.length <= 100`; ký tự có mã ASCII trong `[33, 122]`; không có `"` hay `\`.

Nhận xét ví dụ 1: vị trí của `-` (index 2) vẫn là `-`. Các chữ `a b c d` đọc ngược thành `d c b a` và lấp vào các "ô chữ" theo thứ tự trái sang phải.

Lưu ý ví dụ 3: chữ số `1` **không phải chữ cái** nên đứng yên.

In [1]:
s = "ab-cd"
for i, ch in enumerate(s):
    print(i, repr(ch), "chữ cái" if ch.isalpha() else "giữ nguyên")

0 'a' chữ cái
1 'b' chữ cái
2 '-' giữ nguyên
3 'c' chữ cái
4 'd' chữ cái


## 2. Nền tảng Python cần dùng

### 2.1. `enumerate` và `repr`

- `enumerate(s)` cho cặp `(index, ký tự)` — vừa cần vị trí vừa cần giá trị.
- `repr(ch)` in ký tự kèm dấu nháy, giúp thấy rõ ký tự đặc biệt.

### 2.2. `str.isalpha()` — kiểm tra chữ cái

`ch.isalpha()` trả `True` nếu `ch` là chữ cái. Với constraints (ASCII 33..122) thì chữ cái chỉ là `A-Z`, `a-z`.

Lưu ý: ký tự ASCII `[`, `]`, `^`, `_`, `` ` `` (mã 91..96) nằm **giữa** `Z` (90) và `a` (97) — không phải chữ cái. Vì vậy so sánh `'A' <= ch <= 'z'` là **sai**.

In [2]:
for ch in ["a", "Z", "1", "-", "=", "!", "_", "^", "["]:
    print(repr(ch), "isalpha =", ch.isalpha(), "| ord =", ord(ch), "| 'A'<=ch<='z':", "A" <= ch <= "z")

'a' isalpha = True | ord = 97 | 'A'<=ch<='z': True
'Z' isalpha = True | ord = 90 | 'A'<=ch<='z': True
'1' isalpha = False | ord = 49 | 'A'<=ch<='z': False
'-' isalpha = False | ord = 45 | 'A'<=ch<='z': False
'=' isalpha = False | ord = 61 | 'A'<=ch<='z': False
'!' isalpha = False | ord = 33 | 'A'<=ch<='z': False
'_' isalpha = False | ord = 95 | 'A'<=ch<='z': True
'^' isalpha = False | ord = 94 | 'A'<=ch<='z': True
'[' isalpha = False | ord = 91 | 'A'<=ch<='z': True


`ord(ch)` trả mã số của ký tự. Cột cuối cho thấy `'_'` và `'^'` lọt vào khoảng `'A'..'z'` dù không phải chữ — đây là bẫy kinh điển.

Nếu muốn chỉ chấp nhận chữ cái tiếng Anh (an toàn cả với Unicode như `'é'`), có thể viết `ch.isascii() and ch.isalpha()` hoặc `ch in string.ascii_letters`.

In [3]:
import string
print(string.ascii_letters)
print("é".isalpha(), "é" in string.ascii_letters)

abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ
True False


### 2.3. Chuỗi là bất biến (immutable)

Trong Python, **không thể** gán `s[0] = 'x'`. Muốn sửa từng vị trí, ta:
1. Đổi sang list: `chars = list(s)`.
2. Sửa trên list.
3. Ghép lại: `"".join(chars)`.

In [4]:
s = "abc"
try:
    s[0] = "x"
except TypeError as e:
    print("TypeError:", e)

chars = list(s)
chars[0] = "x"
print(chars, "->", "".join(chars))

TypeError: 'str' object does not support item assignment
['x', 'b', 'c'] -> xbc


### 2.4. Slicing đảo ngược `[::-1]`

`seq[start:end:step]` — với `step = -1` ta đi ngược. `"abcd"[::-1] == "dcba"`.

Đây là cách đảo **toàn bộ** chuỗi; bài của ta khó hơn vì phải **bỏ qua** ký tự không phải chữ.

In [5]:
print("abcd"[::-1])
print("ab-cd"[::-1], "<- đảo toàn bộ thì '-' chạy chỗ khác? Ở ví dụ này '-' ở giữa nên trùng hợp đứng yên")
print("a-bcd"[::-1], "<- '-' từ index 1 chạy sang index 3: SAI yêu cầu")

dcba
dc-ba <- đảo toàn bộ thì '-' chạy chỗ khác? Ở ví dụ này '-' ở giữa nên trùng hợp đứng yên
dcb-a <- '-' từ index 1 chạy sang index 3: SAI yêu cầu


### 2.5. Hoán đổi hai phần tử bằng gán đồng thời

`a[i], a[j] = a[j], a[i]` — vế phải tính xong trước rồi mới gán, nên đổi chỗ an toàn không cần biến tạm.

In [6]:
a = list("abcd")
i, j = 0, 3
a[i], a[j] = a[j], a[i]
print(a)

['d', 'b', 'c', 'a']


## 3. Cách trực giác — gom chữ cái, đảo, rồi đặt lại

Ý tưởng tự nhiên:
1. Lấy ra **danh sách các chữ cái** theo thứ tự: `"a-bC-dEf-ghIj"` → `a b C d E f g h I j`.
2. Đảo ngược danh sách đó.
3. Duyệt `s` lại: gặp chữ cái thì lấy chữ tiếp theo trong danh sách đã đảo; gặp ký tự khác thì giữ nguyên.

In [7]:
def reverse_only_letters_collect(s: str) -> str:
    letters = [ch for ch in s if ch.isalpha()]   # bước 1
    letters.reverse()                            # bước 2 (đảo tại chỗ)
    result = []
    k = 0                                        # chỉ số trong letters
    for ch in s:                                 # bước 3
        if ch.isalpha():
            result.append(letters[k])
            k += 1
        else:
            result.append(ch)
    return "".join(result)

for s in ["ab-cd", "a-bC-dEf-ghIj", "Test1ng-Leet=code-Q!"]:
    print(f"{s!r:24} -> {reverse_only_letters_collect(s)!r}")

'ab-cd'                  -> 'dc-ba'
'a-bC-dEf-ghIj'          -> 'j-Ih-gfE-dCba'
'Test1ng-Leet=code-Q!'   -> 'Qedo1ct-eeLg=ntse-T!'


### 3.1. Giải thích list comprehension

`[ch for ch in s if ch.isalpha()]` đọc là: "với mỗi `ch` trong `s`, **nếu** là chữ cái thì đưa `ch` vào list". Tương đương:

```python
letters = []
for ch in s:
    if ch.isalpha():
        letters.append(ch)
```

`list.reverse()` đảo **tại chỗ** và trả `None`; còn `letters[::-1]` tạo list mới.

In [8]:
letters = [ch for ch in "a-bC-dE" if ch.isalpha()]
print(letters)
print(letters.reverse(), "<- reverse() trả None")
print(letters, "<- nhưng list đã bị đảo")

['a', 'b', 'C', 'd', 'E']
None <- reverse() trả None
['E', 'd', 'C', 'b', 'a'] <- nhưng list đã bị đảo


## 4. Nút thắt và insight

Cách gom chữ cái **đúng** và `O(n)`, nhưng tốn thêm list `letters` và list `result` (bộ nhớ `O(n)` phụ). Và nó chưa thể hiện bản chất "đảo ngược".

**Insight:** đảo ngược một dãy thực chất là **đổi chỗ phần tử đầu với phần tử cuối**, rồi thu hẹp vào trong. Đây là kỹ thuật **hai con trỏ** (two pointers):
- `left` bắt đầu từ đầu, `right` từ cuối.
- Nếu `s[left]` không phải chữ → bỏ qua (`left += 1`).
- Nếu `s[right]` không phải chữ → bỏ qua (`right -= 1`).
- Khi cả hai đều là chữ → **đổi chỗ**, rồi cùng tiến vào trong.
- Dừng khi `left >= right`.

Ký tự không phải chữ không bao giờ bị đụng → tự động **đứng yên**.

In [9]:
# Đảo ngược thường bằng hai con trỏ (chưa bỏ qua gì) — để làm quen
def reverse_two_pointers(s):
    a = list(s)
    left, right = 0, len(a) - 1
    while left < right:
        a[left], a[right] = a[right], a[left]
        left += 1
        right -= 1
    return "".join(a)

print(reverse_two_pointers("abcde"), reverse_two_pointers("ab"), reverse_two_pointers("x"))

edcba ba x


## 5. Lời giải chính — hai con trỏ có bỏ qua

Ghép từng mảnh:

**Mảnh 1 — chuẩn bị:** `chars = list(s)`, `left = 0`, `right = len(chars) - 1`.

**Mảnh 2 — vòng lặp chính:** `while left < right:`.

**Mảnh 3 — bỏ qua ký tự không phải chữ:** dùng `if/elif`: nếu `chars[left]` không phải chữ → `left += 1`; nếu `chars[right]` không phải chữ → `right -= 1`.

**Mảnh 4 — cả hai là chữ:** đổi chỗ rồi `left += 1`, `right -= 1`.

**Mảnh 5 — ghép:** `return "".join(chars)`.

Dùng `if/elif/else` giúp **mỗi vòng chỉ làm một việc**, nên điều kiện `left < right` luôn được kiểm tra lại trước khi đụng tới chỉ số.

In [10]:
def reverse_only_letters_two_pointers(s: str) -> str:
    chars = list(s)
    left, right = 0, len(chars) - 1
    while left < right:
        if not chars[left].isalpha():
            left += 1
        elif not chars[right].isalpha():
            right -= 1
        else:
            chars[left], chars[right] = chars[right], chars[left]
            left += 1
            right -= 1
    return "".join(chars)

for s in ["ab-cd", "a-bC-dEf-ghIj", "Test1ng-Leet=code-Q!"]:
    print(f"{s!r:24} -> {reverse_only_letters_two_pointers(s)!r}")

'ab-cd'                  -> 'dc-ba'
'a-bC-dEf-ghIj'          -> 'j-Ih-gfE-dCba'
'Test1ng-Leet=code-Q!'   -> 'Qedo1ct-eeLg=ntse-T!'


`f"{s!r:24}"`: `!r` nghĩa là dùng `repr(s)` (in kèm nháy), `:24` căn rộng 24 ký tự cho thẳng cột.

## 6. TRACE từng bước

Ví dụ `s = "a-bC-d"` (index 0..5: `a - b C - d`).

| Vòng | left | right | chars[left] | chars[right] | Hành động | chars sau |
|:---:|:---:|:---:|:---:|:---:|---|---|
| 1 | 0 | 5 | a | d | đổi chỗ | `d-bC-a` |
| 2 | 1 | 4 | - | - | left không phải chữ → left=2 | `d-bC-a` |
| 3 | 2 | 4 | b | - | right không phải chữ → right=3 | `d-bC-a` |
| 4 | 2 | 3 | b | C | đổi chỗ | `d-Cb-a` |
| dừng | 3 | 2 | | | left ≥ right | `d-Cb-a` |

In [11]:
def trace_reverse_only_letters(s: str) -> str:
    chars = list(s)
    left, right = 0, len(chars) - 1
    step = 1
    print(f"Bắt đầu: {''.join(chars)!r}")
    while left < right:
        L, R = chars[left], chars[right]
        if not L.isalpha():
            action = f"{L!r} không phải chữ -> left += 1"
            left += 1
        elif not R.isalpha():
            action = f"{R!r} không phải chữ -> right -= 1"
            right -= 1
        else:
            chars[left], chars[right] = R, L
            action = f"đổi {L!r} <-> {R!r}"
            left += 1
            right -= 1
        print(f"vòng {step}: {action:32} | left={left}, right={right} | {''.join(chars)!r}")
        step += 1
    print("Kết quả:", "".join(chars))
    return "".join(chars)

trace_reverse_only_letters("a-bC-d")

Bắt đầu: 'a-bC-d'
vòng 1: đổi 'a' <-> 'd'                  | left=1, right=4 | 'd-bC-a'
vòng 2: '-' không phải chữ -> left += 1  | left=2, right=4 | 'd-bC-a'
vòng 3: '-' không phải chữ -> right -= 1 | left=2, right=3 | 'd-bC-a'
vòng 4: đổi 'b' <-> 'C'                  | left=3, right=2 | 'd-Cb-a'
Kết quả: d-Cb-a


'd-Cb-a'

In [12]:
# Ví dụ có ký tự không phải chữ ở hai đầu và chữ số
trace_reverse_only_letters("7_ab-c!")

Bắt đầu: '7_ab-c!'
vòng 1: '7' không phải chữ -> left += 1  | left=1, right=6 | '7_ab-c!'
vòng 2: '_' không phải chữ -> left += 1  | left=2, right=6 | '7_ab-c!'
vòng 3: '!' không phải chữ -> right -= 1 | left=2, right=5 | '7_ab-c!'
vòng 4: đổi 'a' <-> 'c'                  | left=3, right=4 | '7_cb-a!'
vòng 5: '-' không phải chữ -> right -= 1 | left=3, right=3 | '7_cb-a!'
Kết quả: 7_cb-a!


'7_cb-a!'

## 7. Các lời giải khác và bảng so sánh

### 7.1. Cách Stack (ngăn xếp)

**Stack** là cấu trúc "vào sau ra trước" (LIFO). Trong Python, `list` làm stack: `append` để đẩy vào, `pop()` để lấy phần tử **cuối**.

Đẩy mọi chữ cái vào stack; duyệt lại `s`, gặp chữ cái thì `pop()` — vì stack trả theo thứ tự ngược, ta tự nhiên có chữ cái đảo ngược.

In [13]:
stack = []
for ch in "abc":
    stack.append(ch)
    print("push", ch, "->", stack)
while stack:
    print("pop ->", stack.pop(), "| còn", stack)

push a -> ['a']
push b -> ['a', 'b']
push c -> ['a', 'b', 'c']
pop -> c | còn ['a', 'b']
pop -> b | còn ['a']
pop -> a | còn []


In [14]:
def reverse_only_letters_stack(s: str) -> str:
    stack = [ch for ch in s if ch.isalpha()]
    return "".join(stack.pop() if ch.isalpha() else ch for ch in s)

for s in ["ab-cd", "a-bC-dEf-ghIj", "Test1ng-Leet=code-Q!"]:
    print(f"{s!r:24} -> {reverse_only_letters_stack(s)!r}")

'ab-cd'                  -> 'dc-ba'
'a-bC-dEf-ghIj'          -> 'j-Ih-gfE-dCba'
'Test1ng-Leet=code-Q!'   -> 'Qedo1ct-eeLg=ntse-T!'


`X if cond else Y` là **biểu thức điều kiện** (ternary). Generator `(... for ch in s)` truyền thẳng vào `join` — không cần tạo list trung gian.

### 7.2. Cách chỉ số chạy từ cuối (một con trỏ "đọc ngược")

Thay vì stack, giữ con trỏ `j` chạy **từ cuối** `s`, mỗi khi cần một chữ cái thì lùi `j` tới chữ cái gần nhất. Không cần list phụ cho chữ cái.

In [15]:
def reverse_only_letters_back_pointer(s: str) -> str:
    result = []
    j = len(s) - 1
    for ch in s:
        if ch.isalpha():
            while not s[j].isalpha():
                j -= 1
            result.append(s[j])
            j -= 1
        else:
            result.append(ch)
    return "".join(result)

for s in ["ab-cd", "a-bC-dEf-ghIj", "Test1ng-Leet=code-Q!"]:
    print(f"{s!r:24} -> {reverse_only_letters_back_pointer(s)!r}")

'ab-cd'                  -> 'dc-ba'
'a-bC-dEf-ghIj'          -> 'j-Ih-gfE-dCba'
'Test1ng-Leet=code-Q!'   -> 'Qedo1ct-eeLg=ntse-T!'


### 7.3. Cách dùng iterator `reversed` + `next`

`iter_letters = (c for c in reversed(s) if c.isalpha())` là **iterator** sinh các chữ cái từ cuối về đầu. `next(it)` lấy phần tử kế tiếp.

In [16]:
it = (c for c in reversed("a-bC") if c.isalpha())
print(next(it), next(it), next(it))

def reverse_only_letters_iter(s: str) -> str:
    it = (c for c in reversed(s) if c.isalpha())
    return "".join(next(it) if ch.isalpha() else ch for ch in s)

print(reverse_only_letters_iter("Test1ng-Leet=code-Q!"))

C b a
Qedo1ct-eeLg=ntse-T!


### 7.4. Bảng so sánh

`n = len(s)`.

| Cách | Ý tưởng | Time | Space phụ | Nhận xét |
|---|---|---|---|---|
| Gom + đảo + đặt lại | 3 bước rõ ràng | `O(n)` | `O(n)` | dễ hiểu nhất |
| Stack | pop trả ngược | `O(n)` | `O(n)` | ngắn gọn |
| Con trỏ đọc ngược | j lùi từ cuối | `O(n)` | `O(n)` cho kết quả | không cần list chữ cái |
| Iterator reversed | `next(it)` | `O(n)` | `O(n)` cho kết quả | rất "Pythonic" |
| **Hai con trỏ hoán đổi** | đổi đầu-cuối, bỏ qua | `O(n)` | `O(n)` (list chars, bắt buộc vì str bất biến) | chuẩn phỏng vấn, tại chỗ trên mảng |

Ghi chú: trong ngôn ngữ có chuỗi/mảng ký tự sửa được (C++, hoặc khi input là list), hai con trỏ chỉ tốn `O(1)` bộ nhớ phụ.

## 8. Code nộp LeetCode

In [17]:
class Solution:
    def reverseOnlyLetters(self, s: str) -> str:
        chars = list(s)
        left, right = 0, len(chars) - 1
        while left < right:
            if not chars[left].isalpha():
                left += 1
            elif not chars[right].isalpha():
                right -= 1
            else:
                chars[left], chars[right] = chars[right], chars[left]
                left += 1
                right -= 1
        return "".join(chars)

sol = Solution()
assert sol.reverseOnlyLetters("ab-cd") == "dc-ba"
assert sol.reverseOnlyLetters("a-bC-dEf-ghIj") == "j-Ih-gfE-dCba"
assert sol.reverseOnlyLetters("Test1ng-Leet=code-Q!") == "Qedo1ct-eeLg=ntse-T!"
print("Solution PASS 3 ví dụ đề bài")

Solution PASS 3 ví dụ đề bài


## 9. Vì sao thuật toán đúng?

Gọi `p_0 < p_1 < ... < p_{m-1}` là các vị trí chữ cái trong `s`. Yêu cầu: chữ ở `p_t` trong kết quả = chữ ở `p_{m-1-t}` trong `s`. Nói cách khác, cần **đổi chỗ cặp (p_t, p_{m-1-t})** với mọi `t < m/2`.

**Invariant:** ở đầu mỗi vòng lặp,
1. Mọi vị trí `< left` và `> right` đã ở trạng thái cuối cùng đúng.
2. Số chữ cái ở `[0, left)` bằng số chữ cái ở `(right, n-1]` — tức là `left` và `right` đang "đi cặp" đối xứng theo thứ tự chữ cái.

- Bỏ qua ký tự không phải chữ: không đổi số chữ cái hai bên → invariant giữ.
- Đổi chỗ hai chữ cái: đó chính là cặp `(p_t, p_{m-1-t})` tiếp theo → đặt đúng 2 vị trí, invariant giữ.
- Khi `left >= right`: mọi cặp đã đổi; nếu `m` lẻ, chữ cái giữa tự đổi với chính nó (đứng yên) — vẫn đúng.

Ký tự không phải chữ không bao giờ bị gán → đứng yên. ∎

**Kết thúc:** mỗi vòng `right - left` giảm ít nhất 1 → tối đa `n` vòng.

In [18]:
# Kiểm chứng invariant 2 bằng assert trong lúc chạy
def checked_two_pointers(s):
    chars = list(s)
    left, right = 0, len(chars) - 1
    while left < right:
        n_left = sum(c.isalpha() for c in s[:left])
        n_right = sum(c.isalpha() for c in s[right + 1:])
        assert n_left == n_right, (left, right)
        if not chars[left].isalpha():
            left += 1
        elif not chars[right].isalpha():
            right -= 1
        else:
            chars[left], chars[right] = chars[right], chars[left]
            left += 1
            right -= 1
    return "".join(chars)

print(checked_two_pointers("Test1ng-Leet=code-Q!"), "- invariant luôn đúng")

Qedo1ct-eeLg=ntse-T! - invariant luôn đúng


`sum(c.isalpha() for c in ...)` hoạt động vì `True` được tính là `1`, `False` là `0`.

## 10. Phân tích Time / Space

Gọi `n = len(s)`.

- **Time:** mỗi vòng lặp `left` tăng hoặc `right` giảm (hoặc cả hai) → tối đa `n` vòng, mỗi vòng `O(1)`. Cộng `list(s)` và `join` đều `O(n)`. Tổng `O(n)`.
- **Space:** list `chars` có `n` phần tử → `O(n)`. Trong Python không tránh được vì `str` bất biến. Ngoài `chars`, thuật toán chỉ dùng `O(1)` (hai biến chỉ số).

## 11. Bộ test — mẫu đề, biên, phản ví dụ, đối chiếu

Nhóm test:
1. Ba ví dụ đề.
2. Biên: 1 ký tự chữ, 1 ký tự không phải chữ, toàn ký tự không phải chữ.
3. Toàn chữ cái → giống đảo thường.
4. Ký tự `_ ^ [ ]` (bẫy `'A'..'z'`).
5. Ký tự không phải chữ ở hai đầu.

In [19]:
tests = [
    ("ab-cd", "dc-ba"),
    ("a-bC-dEf-ghIj", "j-Ih-gfE-dCba"),
    ("Test1ng-Leet=code-Q!", "Qedo1ct-eeLg=ntse-T!"),
    ("a", "a"),
    ("-", "-"),
    ("7_=!", "7_=!"),
    ("abcde", "edcba"),
    ("a_b^c", "c_b^a"),
    ("!ab?", "!ba?"),
    ("[z]", "[z]"),
    ("ab[]cd", "dc[]ba"),
]
solvers = [reverse_only_letters_collect, reverse_only_letters_two_pointers,
           reverse_only_letters_stack, reverse_only_letters_back_pointer,
           reverse_only_letters_iter, Solution().reverseOnlyLetters]
for s, expected in tests:
    for f in solvers:
        got = f(s)
        assert got == expected, (f.__name__, s, got, expected)
print(f"PASS {len(tests)} test x {len(solvers)} cách")

PASS 11 test x 6 cách


In [20]:
# Kiểm thử ngẫu nhiên có seed + kiểm tra 2 tính chất của output
import random
rng = random.Random(917)
alphabet = [chr(c) for c in range(33, 123) if chr(c) not in '"\\']
for _ in range(500):
    s = "".join(rng.choice(alphabet) for _ in range(rng.randint(1, 30)))
    out = reverse_only_letters_two_pointers(s)
    # tính chất 1: ký tự không phải chữ đứng yên
    assert all(out[i] == s[i] for i in range(len(s)) if not s[i].isalpha())
    # tính chất 2: dãy chữ cái của out = dãy chữ cái của s đảo ngược
    assert [c for c in out if c.isalpha()] == [c for c in s if c.isalpha()][::-1]
    for f in solvers:
        assert f(s) == out
print("PASS 500 test ngẫu nhiên")

PASS 500 test ngẫu nhiên


## 12. Lỗi người mới hay gặp

### Lỗi 1: Kiểm tra chữ cái bằng khoảng `'A' <= ch <= 'z'`

In [21]:
def wrong_range_check(s):
    chars = list(s)
    is_letter = lambda c: "A" <= c <= "z"
    l, r = 0, len(chars) - 1
    while l < r:
        if not is_letter(chars[l]):
            l += 1
        elif not is_letter(chars[r]):
            r -= 1
        else:
            chars[l], chars[r] = chars[r], chars[l]
            l += 1; r -= 1
    return "".join(chars)

print(wrong_range_check("a_bc"), "| đúng:", reverse_only_letters_two_pointers("a_bc"))

cb_a | đúng: c_ba


**Vì sao sai:** `_` có mã 95, nằm giữa `Z`(90) và `a`(97). **Sửa:** dùng `isalpha()` hoặc `('a' <= c <= 'z') or ('A' <= c <= 'Z')`.

### Lỗi 2: Dùng hai vòng `while` lồng mà quên kiểm tra `left < right`

In [22]:
def wrong_inner_while(s):
    chars = list(s)
    l, r = 0, len(chars) - 1
    while l < r:
        while not chars[l].isalpha():   # thiếu điều kiện l < r
            l += 1
        while not chars[r].isalpha():
            r -= 1
        chars[l], chars[r] = chars[r], chars[l]
        l += 1; r -= 1
    return "".join(chars)

try:
    print(wrong_inner_while("--"))
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


**Vì sao sai:** với chuỗi toàn ký tự không phải chữ, `l` chạy vượt khỏi mảng. **Sửa:** `while l < r and not chars[l].isalpha():` — hoặc dùng `if/elif/else` như lời giải chính.

### Lỗi 3: Gán trực tiếp vào chuỗi

`s[left] = s[right]` gây `TypeError` (đã thấy ở 2.3). **Sửa:** làm trên `list(s)` rồi `join`.

### Lỗi 4: Đổi chỗ xong quên dịch con trỏ → lặp vô hạn

Nếu sau khi đổi chỗ không `left += 1; right -= 1`, vòng lặp đổi đi đổi lại mãi. Ta minh họa bằng cách giới hạn số vòng.

In [23]:
def wrong_no_move(s, max_steps=6):
    chars = list(s)
    l, r = 0, len(chars) - 1
    steps = 0
    while l < r and steps < max_steps:
        if not chars[l].isalpha():
            l += 1
        elif not chars[r].isalpha():
            r -= 1
        else:
            chars[l], chars[r] = chars[r], chars[l]   # quên dịch l, r
        steps += 1
    return "".join(chars), steps

print(wrong_no_move("ab"), "<- hết 6 bước mà vẫn kẹt ở l=0, r=1")

('ab', 6) <- hết 6 bước mà vẫn kẹt ở l=0, r=1


### Lỗi 5: Đảo toàn bộ chuỗi rồi "sửa" ký tự đặc biệt

`s[::-1]` làm ký tự đặc biệt chạy chỗ; cố gắng đặt lại sau đó rất dễ sai. Hãy chọn một trong các cách ở mục 3, 5, 7.

## 13. Mẹo nhận dạng pattern

- "Đảo ngược / đối xứng" + "bỏ qua một số ký tự" → **hai con trỏ từ hai đầu**, mỗi vòng xử lý: bỏ qua bên trái, bỏ qua bên phải, hoặc xử lý cặp.
- "Giữ nguyên vị trí của loại A, sắp xếp lại loại B" → gom loại B ra, xử lý, rồi **đổ ngược lại vào các ô của loại B**.
- Python `str` bất biến → luôn `list(s)` ... `"".join(...)`.

Bài cùng khuôn: 344 Reverse String, 345 Reverse Vowels of a String, 125 Valid Palindrome, 680 Valid Palindrome II, 2810 Faulty Keyboard.

## 14. Biến thể

### 14.1. Reverse Vowels (LeetCode 345)

Chỉ đảo các **nguyên âm** `aeiouAEIOU`. Chỉ cần đổi hàm điều kiện.

In [24]:
def reverse_by_predicate(s: str, keep_moving) -> str:
    chars = list(s)
    l, r = 0, len(chars) - 1
    while l < r:
        if not keep_moving(chars[l]):
            l += 1
        elif not keep_moving(chars[r]):
            r -= 1
        else:
            chars[l], chars[r] = chars[r], chars[l]
            l += 1; r -= 1
    return "".join(chars)

VOWELS = set("aeiouAEIOU")
print(reverse_by_predicate("hello", lambda c: c in VOWELS))      # holle
print(reverse_by_predicate("leetcode", lambda c: c in VOWELS))   # leotcede

holle
leotcede


Ở đây ta truyền **hàm** làm tham số (`keep_moving`) — kỹ thuật tổng quát hóa: một khung hai con trỏ, nhiều bài.

### 14.2. Chỉ đảo chữ số, giữ chữ cái

In [25]:
print(reverse_by_predicate("a1b2c3", str.isdigit))   # a3b2c1

a3b2c1


### 14.3. Kiểm tra palindrome bỏ qua ký tự không phải chữ/số (LeetCode 125)

Cùng khung hai con trỏ, nhưng thay "đổi chỗ" bằng "so sánh".

In [26]:
def is_palindrome_alnum(s: str) -> bool:
    l, r = 0, len(s) - 1
    while l < r:
        if not s[l].isalnum():
            l += 1
        elif not s[r].isalnum():
            r -= 1
        else:
            if s[l].lower() != s[r].lower():
                return False
            l += 1; r -= 1
    return True

print(is_palindrome_alnum("A man, a plan, a canal: Panama"), is_palindrome_alnum("race a car"))

True False


## 15. Bài tập tự luyện (TODO — không tự chạy)

### Bài 1 — Đoán output
```python
"x_Y".isalpha(), "_".isalpha(), "Z" < "_" < "a", "".join(["a", "-", "b"][::-1])
```

### Bài 2 — Đảo chuỗi thường bằng hai con trỏ (LeetCode 344)
Đảo **tại chỗ** một list ký tự, không dùng `[::-1]` hay `reverse()`.

In [27]:
def exercise_reverse_list(chars: list[str]) -> None:
    # TODO: hai con trỏ, đổi chỗ, không return gì
    pass

# a = list("hello"); exercise_reverse_list(a); assert a == list("olleh")

### Bài 3 — Tự viết lại lời giải 917 bằng hai con trỏ

In [28]:
def exercise_reverse_only_letters(s: str) -> str:
    # TODO 1: chars = list(s), left, right
    # TODO 2: while left < right: bỏ qua trái / bỏ qua phải / đổi chỗ
    # TODO 3: join
    pass

# assert exercise_reverse_only_letters("ab-cd") == "dc-ba"
# assert exercise_reverse_only_letters("a-bC-dEf-ghIj") == "j-Ih-gfE-dCba"

### Bài 4 — Reverse Vowels (LeetCode 345)

In [29]:
def exercise_reverse_vowels(s: str) -> str:
    # TODO
    pass

# assert exercise_reverse_vowels("hello") == "holle"
# assert exercise_reverse_vowels("IceCreAm") == "AceCreIm"

### Bài 5 — Đảo chữ cái trong **từng từ**, giữ thứ tự các từ

Ví dụ: `"ab-c de!f"` → mỗi từ (tách bằng dấu cách) đảo chữ cái riêng: `"cb-a fe!d"`.

In [30]:
def exercise_reverse_letters_each_word(s: str) -> str:
    # TODO: tách theo ' ', áp dụng lời giải 917 cho từng từ, ghép lại bằng ' '
    pass

# assert exercise_reverse_letters_each_word("ab-c de!f") == "cb-a fe!d"

### Bài 6 — Mini challenge: đảo chữ cái nhưng **giữ vị trí hoa/thường**

Chữ cái đảo ngược, nhưng vị trí nào ban đầu là chữ hoa thì kết quả ở đó vẫn là chữ hoa. Ví dụ `"Ab-c"` → chữ cái đảo: `c b A` → áp kiểu hoa/thường theo vị trí `A b c` (Hoa, thường, thường) → `"Cb-a"`.

In [31]:
def exercise_reverse_keep_case(s: str) -> str:
    # TODO
    pass

# assert exercise_reverse_keep_case("Ab-c") == "Cb-a"
# assert exercise_reverse_keep_case("a-B") == "b-A"

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | `_` không phải chữ; so sánh chuỗi theo `ord` | `ord('Z')=90, ord('_')=95, ord('a')=97` | join list đảo |
| 2 | đổi đầu-cuối rồi thu hẹp | `while l < r` | `chars[l], chars[r] = chars[r], chars[l]` |
| 3 | khung if / elif / else | điều kiện `isalpha()` | xem mục 5 |
| 4 | như 917, đổi điều kiện | `VOWELS = set("aeiouAEIOU")` | `c in VOWELS` |
| 5 | `s.split(" ")` | tái dùng hàm Bài 3 | `" ".join(f(w) for w in ...)` |
| 6 | đảo trước, sửa hoa/thường sau | nhớ vị trí hoa của `s` | `c.upper() if s[i].isupper() else c.lower()` |

## 17. Đáp án tham khảo

Bài 1: `(False, False, True, 'b-a')`.

In [32]:
print(("x_Y".isalpha(), "_".isalpha(), "Z" < "_" < "a", "".join(["a", "-", "b"][::-1])))

(False, False, True, 'b-a')


In [33]:
def answer_reverse_list(chars):
    l, r = 0, len(chars) - 1
    while l < r:
        chars[l], chars[r] = chars[r], chars[l]
        l += 1; r -= 1

def answer_reverse_only_letters(s):
    chars = list(s)
    l, r = 0, len(chars) - 1
    while l < r:
        if not chars[l].isalpha():
            l += 1
        elif not chars[r].isalpha():
            r -= 1
        else:
            chars[l], chars[r] = chars[r], chars[l]
            l += 1; r -= 1
    return "".join(chars)

def answer_reverse_vowels(s):
    vowels = set("aeiouAEIOU")
    chars = list(s)
    l, r = 0, len(chars) - 1
    while l < r:
        if chars[l] not in vowels:
            l += 1
        elif chars[r] not in vowels:
            r -= 1
        else:
            chars[l], chars[r] = chars[r], chars[l]
            l += 1; r -= 1
    return "".join(chars)

def answer_reverse_letters_each_word(s):
    return " ".join(answer_reverse_only_letters(w) for w in s.split(" "))

def answer_reverse_keep_case(s):
    rev = answer_reverse_only_letters(s)
    return "".join(c.upper() if s[i].isupper() else c.lower() if c.isalpha() else c
                   for i, c in enumerate(rev))

a = list("hello"); answer_reverse_list(a); assert a == list("olleh")
assert answer_reverse_only_letters("ab-cd") == "dc-ba"
assert answer_reverse_only_letters("a-bC-dEf-ghIj") == "j-Ih-gfE-dCba"
assert answer_reverse_vowels("hello") == "holle"
assert answer_reverse_vowels("IceCreAm") == "AceCreIm"
assert answer_reverse_letters_each_word("ab-c de!f") == "cb-a fe!d"
assert answer_reverse_keep_case("Ab-c") == "Cb-a"
assert answer_reverse_keep_case("a-B") == "b-A"
for s, expected in tests:
    assert answer_reverse_only_letters(s) == expected
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def reverseOnlyLetters(self, s: str) -> str:
        chars = list(s)
        left, right = 0, len(chars) - 1
        while left < right:
            if not chars[left].isalpha():
                left += 1
            elif not chars[right].isalpha():
                right -= 1
            else:
                chars[left], chars[right] = chars[right], chars[left]
                left += 1
                right -= 1
        return "".join(chars)
```

**Khung "hai con trỏ có bỏ qua":** `while l < r: if bỏ_trái: l+=1 / elif bỏ_phải: r-=1 / else: xử lý cặp; l+=1; r-=1`.

**Checklist trước khi submit:**
- [ ] Dùng `isalpha()` (không dùng khoảng `'A'..'z'`).
- [ ] Làm trên `list(s)`, cuối cùng `"".join`.
- [ ] Sau khi đổi chỗ, dịch **cả hai** con trỏ.
- [ ] Nếu dùng `while` lồng, mỗi vòng con có `left < right`.
- [ ] Test chuỗi toàn ký tự đặc biệt, chuỗi 1 ký tự, chữ số trong chuỗi.